# Stima della potenza ricevuta


Vogliamo dare una stima della potenza ricevuta e del voltaggio che ci aspettiamo sull'oscilloscopio

$$
P_{tot} = k_B G \Delta \nu (T_A+ T_N)
$$
dove:
* G è il guadagno dell'amplificatore
* $ \Delta \nu $ è la banda selezionata dal filtro
* $ T_A $ è la temperatura di antenna
* $ T_N $ è la temperatura del rumore

### Guadagno dell'amplificatore
I dati di targa per i due amplificatori riportano il guadagno in dBi, tramite la formula
$$G = 10 ^ {\frac{G_{dbi}}{10}}$$
valutata a 30 GHz
Per i due amplificatori otteniamo


In [1]:
#importo le librerie
import numpy as np
from scipy.integrate import quad
from scipy.constants import c, Boltzmann,h

In [2]:
G_miteq_dbi = 31.37
G_miteq = 10**(G_miteq_dbi/10)

G_erzia_dbi = 55
G_erzia = 10**(G_erzia_dbi/10)


print("### GAIN ###")
print(f"miteq , {G_miteq}")
print(f"erzia , {G_erzia}")

### GAIN ###
miteq , 1370.8817661648536
erzia , 316227.7660168379


### Calcolo della temperatura di rumore
Calcoliamo la temperatura dela rumore degli amplificatori partendo dalla noise figure:
$$ F = 10^{\frac{N_{dBi}}{10} }$$
$$ T_N = T_0 (F-1) $$
dove $T_0$ è la temperatura a cui è stata effettuata la misura di rumore, calcoliamo per entrambi gli amplificatori

In [5]:
N_miteq_dbi = 2.5
N_miteq = 10**(N_miteq_dbi/10)
T_miteq = 273.15 + 25
TN_miteq = T_miteq*(N_miteq -1)

N_erzia_dbi = 2.75 # from the noise figure
N_erzia = 10**(N_erzia_dbi/10)
T_erzia = 273.15+25
TN_erzia = T_erzia*(N_erzia -1)


print("### T NOISE ###")
print(f"miteq , {TN_miteq}")
print(f"erzia , {TN_erzia}")

### T NOISE ###
miteq , 232.0440061031048
erzia , 263.45997603138403


### Banda del sistema
Confrontiamo la frequenza di cuttoff della guida WR28, $f_{cut} = 21.077 GHz$, abbiamo una banda passante del filtro che è compresa tra $26.5 GHz - 40.0 GHz$ che ci da una banda passante di $\Delta \nu = 13.5 GHz$, essendo il bordo inferiore della banda passante del filtro maggiore della frequenza di cutoff sarà la banda del filtro da inserire nella formula.

In [4]:
delta_nu = 13.5e9

### Temperatura di antenna
La temperatura di antenna è la convoluzione della temperatura di brillanza per il pattern normalizzato legato al sistema di ricezione:
$$
T_A = \frac{\int T_B(\theta,\Phi) P_N(\theta, \Phi) d \theta d \Phi}{\int P_N(\theta, \Phi)d \theta d \Phi}
$$
approssimiamo la $T_B(\theta, \Phi)$ come uniforme e uguale a quello termico, nel caso del sole, $T_B = 5700 K$ per un angolo $\theta = 0.5˚$.
Per l'angolo $\Phi $ integriamo ed essendo costante esce un coefficiente $2\pi$ che si semplifica con la normalizzazione a denominatore nella formula della $T_A$.

Calcoliamo due differenti temprature di antenna una sfruttando tutta l'apertura del telescopio, come se il feed horn non esistesse e la seconda tenendo condo delle dimensioni fisiche del feed horn e della sola luce che lo illumina.

La forma del pattern normalizzato viene presa come una Gaussiana, non vogliamo una tophat per non fare un'approssimazione troppo grossolana:
$$P(\theta, \Phi) = e^{\frac{-\theta^2}{2 \sigma^2}}$$
prendendo una $\theta_{FWHM} = \frac{\lambda}{D}$ vogliamo convertire questa quantità di scala come una larghezza della gaussiana conoscendo la relazione $\theta_{FWHM} = 2 \sqrt{2 \ln 2} \sigma$

Per la seconda configurazione con il feed horn, vediamo come l'apertura della antenna vede lo specchio, non occuperà litero diametro dello specchio ma una sua frazione, che ci darà un nuovo angolo.

In [6]:
frequency = 30e9
wavelenght = c/frequency
d_mirror = 1.473 #diametro dello specchio primario del telescopio
focal_distance = 0.682
d_horn = 0.0584

sun_half_angle = 0.25*np.pi/180 
T_sun = 5700
x = h*frequency/Boltzmann/T_sun
T_b = x/(np.exp(x)-1)*T_sun


def P_N(x,sigma):
    return np.exp(-x**2/(2*sigma**2))

theta_mirror = wavelenght/d_mirror
sigma_mirror = theta_mirror/(2*np.sqrt(2 *np.log(2)))

PN_mirror, error = quad(P_N, -sun_half_angle, sun_half_angle, args=(sigma_mirror))
PN_mirror_norm, error = quad(P_N, -np.pi, np.pi, args=(sigma_mirror))

T_A_mirror = T_b*PN_mirror/PN_mirror_norm


theta_horn = np.arctan(d_horn/focal_distance)
sigma_horn = theta_horn/(2*np.sqrt(2 *np.log(2)))

PN_horn, error = quad(P_N, -sun_half_angle, sun_half_angle, args=(sigma_horn))
PN_horn_norm, error = quad(P_N, -np.pi, np.pi, args=(sigma_horn))

T_A_horn = T_b*PN_horn/PN_horn_norm

print("### ANTENNA TEMPERATURE ###")
print(f"T_b temperatura di brillanza a 30 GHz,{T_b}")
print(f"T_A mirror,{T_A_mirror}")
print(f"T_A horn,{T_A_horn}")

### ANTENNA TEMPERATURE ###
T_b temperatura di brillanza a 30 GHz,5699.280143845725
T_A mirror,4958.994874812684
T_A horn,545.6555976565104


## Risultati finali
Produciamo le quattro configurazioni tra antenne e amplificatori, otteniamo le potenze

In [7]:
P_mirror_miteeq = Boltzmann * G_miteq * delta_nu *(T_A_mirror+TN_miteq)
P_mirror_erzia = Boltzmann * G_erzia * delta_nu *(T_A_mirror+TN_erzia)

P_horn_miteeq = Boltzmann * G_miteq * delta_nu *(T_A_horn+TN_miteq)
P_horn_erzia = Boltzmann * G_erzia * delta_nu *(T_A_horn+TN_erzia)


print("### POWER ###")
print(f"P_mirror_miteq, {P_mirror_miteeq}")
print(f"P_mirror_erzia, {P_mirror_erzia}")
print(f"P_horn_miteq, {P_horn_miteeq}")
print(f"P_horn_erzia, {P_horn_erzia}")

### POWER ###
P_mirror_miteq, 1.3259925622844368e-06
P_mirror_erzia, 0.000305323765893794
P_horn_miteq, 1.9831648726823446e-07
P_horn_erzia, 4.519740431074335e-05
